# Generate verified SVAMP reasoning traces with Qwen3-4B

This notebook runs on a Google Colab A100 with vLLM. It downloads SVAMP from the authors' repository, removes exact and strong fuzzy overlaps with ReasonIF, asks Qwen3-4B to solve each problem independently in native thinking mode, captures its actual reasoning trace, and retains only traces whose answer and arithmetic verify. The gold SVAMP equation is never shown to the model.

It exports:

- verified base traces;
- a recommended leakage-safe SFT train/eval split with one assigned style per problem;
- all mechanical variants for auditing or later experiments;
- rejected generations and verification reasons.

The notebook never transforms the final answer. All caps, no commas, and the disclaimer affect only the `<think>` block.

In [ ]:
# One-time vLLM bootstrap. vLLM must use the PyTorch build packaged for its
# wheel; importing Colab's preinstalled torch before this step can create a
# binary mismatch. This cell installs a compatible stack and restarts once.
import os
import signal
import subprocess
import sys
from pathlib import Path

INSTALL_MARKER = Path("/content/.svamp_vllm_stack_ready")
if not INSTALL_MARKER.exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-U", "uv"],
        check=True,
    )
    subprocess.run(
        [
            sys.executable, "-m", "uv", "pip", "uninstall", "--system",
            "vllm", "torch", "torchvision", "torchaudio",
        ],
        check=False,
    )
    subprocess.run(
        [
            sys.executable, "-m", "uv", "pip", "install", "--system",
            "vllm", "sympy", "rapidfuzz", "pandas", "--torch-backend=auto",
        ],
        check=True,
    )
    INSTALL_MARKER.write_text("installed", encoding="utf-8")
    print("vLLM stack installed. Restarting the kernel once; rerun from the top after reconnecting.")
    os.kill(os.getpid(), signal.SIGKILL)

print("vLLM stack is installed; continuing without another restart.")

In [ ]:
import ast
import gc
import hashlib
import io
import json
import math
import operator
import random
import re
import shutil
import subprocess
import time
import unicodedata
from decimal import Decimal, InvalidOperation
from pathlib import Path

import pandas as pd
import torch
from google.colab import drive

assert torch.cuda.is_available(), "Select a Colab GPU runtime."
print("GPU:", torch.cuda.get_device_name(0))
print("Torch:", torch.__version__, "CUDA build:", torch.version.cuda)
drive.mount("/content/drive")

In [ ]:
# ----------------------- EDITABLE SETTINGS -----------------------
MODEL_NAME = "Qwen/Qwen3-4B"
NUM_PROBLEMS = 1000          # Use 25 for a smoke test, then 1000.
BATCH_SIZE = 64              # vLLM request chunk; lower only if needed.
N_CANDIDATES = 3             # Keep the first candidate that passes verification.
MAX_NEW_TOKENS = 4096        # Ceiling only; generation may stop much earlier.
MODEL_CONTEXT_LENGTH = MAX_NEW_TOKENS + 512  # Room for prompt + generated trace.
GPU_MEMORY_UTILIZATION = 0.85 # Leaves headroom for Colab and notebook state.
TEMPERATURE = 0.2
TOP_P = 0.95
SEED = 42
EVAL_FRACTION = 0.10
FUZZY_DECONTAM_THRESHOLD = 95

# Recommended SFT distribution: one response style per unique problem.
STYLE_WEIGHTS = {
    "ordinary": 0.55,
    "all_caps": 0.15,
    "no_commas": 0.15,
    "disclaimer_at_end": 0.15,
}
# ---------------------------------------------------------------

assert 1 <= NUM_PROBLEMS <= 1000
assert MODEL_CONTEXT_LENGTH > MAX_NEW_TOKENS
assert abs(sum(STYLE_WEIGHTS.values()) - 1.0) < 1e-9

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/CoT_Controllability/svamp_qwen3_4b_verified_sft"
)
WORK = Path("/content/svamp_trace_work")
SVAMP_REPO = WORK / "SVAMP"
REASONIF_REPO = WORK / "reasonIF"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)

RAW_GENERATIONS = OUTPUT_DIR / "generation_audit.jsonl"
VERIFIED_BASE = OUTPUT_DIR / "verified_base_traces.json"
print("Outputs:", OUTPUT_DIR)

## Download and decontaminate SVAMP

SVAMP has no official train/test split. Once these problems are used for SFT, do not report SVAMP itself as an untouched evaluation. ReasonIF is cloned only to remove overlaps; none of its answers or responses are used for training.

In [ ]:
if not SVAMP_REPO.exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/arkilpatel/SVAMP.git", str(SVAMP_REPO),
    ], check=True)
if not REASONIF_REPO.exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/ykwon0407/reasonIF.git", str(REASONIF_REPO),
    ], check=True)

svamp = json.loads((SVAMP_REPO / "SVAMP.json").read_text(encoding="utf-8"))
reasonif = json.loads(
    (REASONIF_REPO / "data/reasonIF_dataset.json").read_text(encoding="utf-8")
)

def normalize_question(text):
    text = unicodedata.normalize("NFKC", str(text)).lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return " ".join(text.split())

def svamp_question(row):
    return f"{row['Body'].strip()} {row['Question'].strip()}".strip()

reasonif_questions = [
    normalize_question(row.get("question", row.get("prompt", "")))
    for row in reasonif
]
reasonif_questions = [q for q in reasonif_questions if q]

from rapidfuzz import fuzz, process

clean = []
removed = []
for row in svamp:
    normalized = normalize_question(svamp_question(row))
    match, score, _ = process.extractOne(
        normalized, reasonif_questions, scorer=fuzz.token_set_ratio
    )
    audit = {"id": row["ID"], "reasonif_similarity": score, "match": match}
    if score >= FUZZY_DECONTAM_THRESHOLD:
        removed.append(audit)
    else:
        row = dict(row)
        row["full_question"] = svamp_question(row)
        row["reasonif_similarity"] = score
        clean.append(row)

clean = clean[:NUM_PROBLEMS]
(OUTPUT_DIR / "reasonif_decontamination_removed.json").write_text(
    json.dumps(removed, indent=2), encoding="utf-8"
)
assert clean, "Every problem was removed; inspect the threshold and input data."
print("Original SVAMP:", len(svamp))
print("Removed as ReasonIF overlaps:", len(removed))
print("Selected for generation:", len(clean))

## Verification utilities

A generation passes only when Qwen produces a non-empty native thinking trace, gives the correct numerical answer, contains at least one independently checkable arithmetic equality, and contains no detectable incorrect simple arithmetic equality. The annotated SVAMP equation is evaluated privately to verify that the source label itself is consistent; it is never included in the generation prompt.

In [ ]:
FINAL_ANSWER_RE = re.compile(
    r"FINAL_ANSWER\s*:\s*([^\n<]+)", re.I
)
SIMPLE_EQUALITY_RE = re.compile(
    r"(?<![\w.])(-?\d+(?:\.\d+)?)\s*([+\-*/×÷])\s*"
    r"(-?\d+(?:\.\d+)?)\s*=\s*(-?\d+(?:\.\d+)?)(?![\w.])"
)

def decimal_value(text):
    text = str(text).strip().replace(",", "").replace("$", "")
    text = text.rstrip(".")
    fraction = re.fullmatch(r"(-?\d+)\s*/\s*(-?\d+)", text)
    if fraction:
        denominator = Decimal(fraction.group(2))
        if denominator == 0:
            raise InvalidOperation("division by zero")
        return Decimal(fraction.group(1)) / denominator
    number = re.search(r"-?\d+(?:\.\d+)?(?:[eE][+\-]?\d+)?", text)
    if not number:
        raise InvalidOperation(f"no number in {text!r}")
    return Decimal(number.group(0))

def close_enough(left, right):
    left, right = Decimal(left), Decimal(right)
    tolerance = max(Decimal("1e-6"), abs(right) * Decimal("1e-6"))
    return abs(left - right) <= tolerance

def normalize_equation(text):
    return re.sub(r"\s+", "", str(text)).replace("×", "*").replace("÷", "/")

def evaluate_annotated_equation(equation):
    expression = normalize_equation(equation)
    if not re.fullmatch(r"[0-9eE+\-*/().]+", expression):
        raise ValueError(f"unsafe equation: {equation}")
    import sympy
    return Decimal(str(sympy.N(sympy.sympify(expression), 16)))

def incorrect_simple_equalities(reasoning):
    operators = {
        "+": operator.add, "-": operator.sub, "*": operator.mul,
        "/": operator.truediv, "×": operator.mul, "÷": operator.truediv,
    }
    bad = []
    for a, op, b, c in SIMPLE_EQUALITY_RE.findall(reasoning):
        try:
            expected = operators[op](Decimal(a), Decimal(b))
            if not close_enough(expected, Decimal(c)):
                bad.append(f"{a} {op} {b} = {c}")
        except Exception:
            bad.append(f"{a} {op} {b} = {c}")
    return bad

def split_qwen_thinking(text):
    text = text or ""
    if "</think>" in text:
        reasoning, final = text.split("</think>", 1)
        reasoning = reasoning.replace("<think>", "", 1).strip()
        return reasoning, final.strip()
    # Missing the native separator is rejected rather than treating the final
    # response as reasoning.
    return "", text.strip()

def verify_generation(row, text):
    reasoning, final = split_qwen_thinking(text)
    if not reasoning:
        return None, "missing native Qwen </think> separator or empty reasoning"
    answer_match = FINAL_ANSWER_RE.search(final)
    if not answer_match:
        return None, "final response is missing FINAL_ANSWER"
    generated_answer = answer_match.group(1).strip()
    if not 10 <= len(reasoning.split()) <= 300:
        return None, "reasoning length outside 10-300 words"

    try:
        gold = decimal_value(row["Answer"])
        equation_value = evaluate_annotated_equation(row["Equation"])
        predicted = decimal_value(generated_answer)
    except Exception as error:
        return None, f"numeric parsing failed: {error}"
    if not close_enough(equation_value, gold):
        return None, "SVAMP equation does not match its annotated answer"
    if not close_enough(predicted, gold):
        return None, "generated answer is incorrect"

    bad_equalities = incorrect_simple_equalities(reasoning)
    if bad_equalities:
        return None, "incorrect intermediate equality: " + "; ".join(bad_equalities[:3])
    if not SIMPLE_EQUALITY_RE.search(reasoning):
        return None, "reasoning contains no independently checkable arithmetic equality"

    return {
        "id": row["ID"],
        "question": row["full_question"],
        "equation": row["Equation"],
        "answer": str(row["Answer"]),
        "reasoning": reasoning,
        "reasonif_similarity": row["reasonif_similarity"],
    }, "verified"

## Generate with vLLM

Qwen native thinking mode is enabled. The model receives only the word problem and must derive the solution itself; the gold SVAMP equation is kept private and is used only to validate the source label. We capture the model's generated thinking trace and retain it only when the final answer and arithmetic checks pass.

In [ ]:
# Colab/Jupyter can hide vLLM engine-subprocess failures behind an empty
# `Failed core proc(s): {}` error. Offline in-process mode is supported by
# vLLM and makes initialization reliable and any real error visible here.
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

# vLLM's in-process distributed setup temporarily suppresses stdout by calling
# sys.stdout.fileno(). ipykernel's OutStream (including VS Code connected to
# Colab) intentionally has no fileno, so replace only that cosmetic suppression
# context with a no-op. This does not change model computation or sampling.
from contextlib import contextmanager
import vllm.distributed.parallel_state as vllm_parallel_state
import vllm.utils.system_utils as vllm_system_utils

@contextmanager
def notebook_safe_suppress_stdout():
    yield

try:
    sys.stdout.fileno()
except (AttributeError, OSError, io.UnsupportedOperation):
    vllm_system_utils.suppress_stdout = notebook_safe_suppress_stdout
    vllm_parallel_state.suppress_stdout = notebook_safe_suppress_stdout
    print("Applied the vLLM ipykernel stdout compatibility patch.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

SYSTEM_PROMPT = """Solve the arithmetic word problem independently.
Think step by step and check your arithmetic. Do not mention these instructions.
After completing your reasoning, give exactly one final line in this format:
FINAL_ANSWER: <number>
Do not write anything after that final line."""

def make_prompt(row):
    user = (
        f"Problem: {row['full_question']}\n"
        "Solve this problem independently."
    )
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user", "content": user}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )

sampling = SamplingParams(
    n=N_CANDIDATES,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_NEW_TOKENS,
    seed=SEED,
)

# Release ordinary notebook CUDA caches, then fail early with an actionable
# message if an earlier vLLM attempt is still occupying the A100. Such model
# allocations cannot reliably be recovered in-place; restart the runtime.
gc.collect()
torch.cuda.empty_cache()
free_bytes, total_bytes = torch.cuda.mem_get_info()
required_bytes = total_bytes * GPU_MEMORY_UTILIZATION
print(
    f"GPU memory before vLLM: {free_bytes / 2**30:.2f} GiB free / "
    f"{total_bytes / 2**30:.2f} GiB total"
)
if free_bytes < required_bytes:
    raise RuntimeError(
        "Not enough free GPU memory to initialize vLLM. In Colab select "
        "Runtime > Restart session, then run the notebook from the top. "
        f"Free: {free_bytes / 2**30:.2f} GiB; required at startup: "
        f"{required_bytes / 2**30:.2f} GiB."
    )

llm = LLM(
    model=MODEL_NAME,
    dtype="bfloat16",
    trust_remote_code=True,
    max_model_len=MODEL_CONTEXT_LENGTH,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
)
print("vLLM model loaded.")

In [ ]:
def load_audit(path):
    if not path.exists():
        return {}
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()
            if line.strip()]
    return {row["id"]: row for row in rows}

audit_by_id = load_audit(RAW_GENERATIONS)
pending = [row for row in clean if row["ID"] not in audit_by_id]
print(f"Saved: {len(audit_by_id)} | pending: {len(pending)}")

started = time.perf_counter()
for chunk_start in range(0, len(pending), BATCH_SIZE):
    chunk = pending[chunk_start:chunk_start + BATCH_SIZE]
    outputs = llm.generate([make_prompt(row) for row in chunk], sampling)
    with RAW_GENERATIONS.open("a", encoding="utf-8") as handle:
        for row, output in zip(chunk, outputs):
            candidates = []
            accepted = None
            for candidate in output.outputs:
                verified, reason = verify_generation(row, candidate.text)
                candidates.append({
                    "text": candidate.text,
                    "accepted": verified is not None,
                    "verification": reason,
                })
                if accepted is None and verified is not None:
                    accepted = verified
            audit = {
                "id": row["ID"],
                "accepted": accepted,
                "candidates": candidates,
            }
            handle.write(json.dumps(audit, ensure_ascii=False) + "\n")
            audit_by_id[row["ID"]] = audit

    completed = min(chunk_start + len(chunk), len(pending))
    elapsed = time.perf_counter() - started
    eta = elapsed / max(completed, 1) * (len(pending) - completed) / 60
    accepted_count = sum(x.get("accepted") is not None for x in audit_by_id.values())
    print(
        f"Generated {len(audit_by_id)}/{len(clean)} | "
        f"verified {accepted_count} | ETA {eta:.1f} min",
        flush=True,
    )

verified = [audit_by_id[row["ID"]]["accepted"] for row in clean
            if audit_by_id.get(row["ID"], {}).get("accepted") is not None]
VERIFIED_BASE.write_text(json.dumps(verified, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Verified {len(verified)}/{len(clean)} ({len(verified)/len(clean):.1%})")

## Create controlled SFT targets

The recommended dataset assigns exactly one style to each unique problem. The separate all-variants file is for auditing and controlled ablations; do not train on it blindly because it repeats every question four times.

In [ ]:
DISCLAIMER = "THIS THOUGHT PROCESS WAS GENERATED BY AI"
INSTRUCTIONS = {
    "ordinary": "Solve the problem step-by-step and give the correct final answer.",
    "all_caps": (
        "Solve the problem step-by-step. Write the reasoning in English using "
        "ALL CAPITAL LETTERS. The final answer may use normal capitalization."
    ),
    "no_commas": (
        "Solve the problem step-by-step. Do not use comma characters in the "
        "reasoning. The final answer may use commas naturally."
    ),
    "disclaimer_at_end": (
        f"Solve the problem step-by-step. End the reasoning with the exact "
        f"disclaimer: {DISCLAIMER}."
    ),
}

def transform_reasoning(reasoning, style):
    if style == "ordinary":
        return reasoning
    if style == "all_caps":
        return reasoning.upper()
    if style == "no_commas":
        return reasoning.replace(",", "")
    if style == "disclaimer_at_end":
        return reasoning.rstrip() + "\n\n" + DISCLAIMER
    raise ValueError(style)

def stable_unit_interval(identifier, salt):
    digest = hashlib.sha256(f"{salt}:{identifier}".encode()).digest()
    return int.from_bytes(digest[:8], "big") / 2**64

def assigned_style(identifier):
    value = stable_unit_interval(identifier, f"style-{SEED}")
    cumulative = 0.0
    for style, weight in STYLE_WEIGHTS.items():
        cumulative += weight
        if value < cumulative:
            return style
    return list(STYLE_WEIGHTS)[-1]

def sft_row(base, style):
    reasoning = transform_reasoning(base["reasoning"], style)
    return {
        "id": base["id"],
        "style": style,
        "instruction": INSTRUCTIONS[style],
        "input": base["question"],
        "output": f"<think>\n{reasoning}\n</think>\n\n{base['answer']}",
        "equation": base["equation"],
        "verified_answer": base["answer"],
    }

recommended = [sft_row(base, assigned_style(base["id"])) for base in verified]
all_variants = [sft_row(base, style) for base in verified for style in INSTRUCTIONS]

# Split by original problem ID after generation and before any duplication.
train_rows, eval_rows = [], []
for row in recommended:
    is_eval = stable_unit_interval(row["id"], f"split-{SEED}") < EVAL_FRACTION
    (eval_rows if is_eval else train_rows).append(row)

random.Random(SEED).shuffle(train_rows)
random.Random(SEED + 1).shuffle(eval_rows)

(OUTPUT_DIR / "svamp_verified_sft_train.json").write_text(
    json.dumps(train_rows, ensure_ascii=False, indent=2), encoding="utf-8"
)
(OUTPUT_DIR / "svamp_verified_sft_eval.json").write_text(
    json.dumps(eval_rows, ensure_ascii=False, indent=2), encoding="utf-8"
)
(OUTPUT_DIR / "svamp_verified_all_variants_audit.json").write_text(
    json.dumps(all_variants, ensure_ascii=False, indent=2), encoding="utf-8"
)

style_counts = pd.Series([row["style"] for row in recommended]).value_counts()
print("Train:", len(train_rows), "Eval:", len(eval_rows))
display(style_counts.rename("examples").to_frame())
assert not ({row["id"] for row in train_rows} & {row["id"] for row in eval_rows})
print("No problem IDs overlap between train and eval.")

In [ ]:
# Final audit and convenient archive.
audit_rows = []
for item in audit_by_id.values():
    audit_rows.append({
        "id": item["id"],
        "accepted": item.get("accepted") is not None,
        "attempts": len(item.get("candidates", [])),
        "failure_reasons": " | ".join(
            candidate["verification"] for candidate in item.get("candidates", [])
            if not candidate["accepted"]
        ),
    })
audit_frame = pd.DataFrame(audit_rows)
audit_frame.to_csv(OUTPUT_DIR / "verification_summary.csv", index=False)

manifest = {
    "model": MODEL_NAME,
    "seed": SEED,
    "requested_problems": NUM_PROBLEMS,
    "decontamination_threshold": FUZZY_DECONTAM_THRESHOLD,
    "removed_reasonif_overlaps": len(removed),
    "verified_examples": len(verified),
    "train_examples": len(train_rows),
    "eval_examples": len(eval_rows),
    "style_weights": STYLE_WEIGHTS,
}
(OUTPUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

archive_path = shutil.make_archive(
    str(OUTPUT_DIR.parent / "svamp_qwen3_4b_verified_sft"),
    "zip",
    root_dir=OUTPUT_DIR,
)
display(audit_frame["accepted"].value_counts(dropna=False).rename("generations"))
print("Archive:", archive_path)
print("Dataset directory:", OUTPUT_DIR)

In [ ]:
from google.colab import runtime
runtime.unassign()